# Stage 13 — Final-test registry ve Stage 04–06 uyumlu seçilmiş dataset

**Amaç:** `validated_v012.json` içindeki *yeni* 300 aday arasından, Stage 12 v2 protokolünün sabit kategori, cevap türü ve zorluk kotalarına uyan 250 soruyu **hedef modellerin herhangi bir yeni cevabını görmeden** seçmek ve SHA-256 ile dondurmak. Eski 1000 soru, 30 Stage-03 sanity sorusu dahil, korunur.

**Stage 04–06 değişikliği:** Algoritmaların hiçbirini değiştirmeyin; yalnız mevcut `VALIDATED_DATASET_VERSION` ayarını **`validated_v012_final250_v1`** olarak seçin. Stage 04'ün Stage-03→04 30-soru reproducibility kontrolü eski soruları gerektirdiği için oluşturulan inference dataset'i `1000 eski + 250 final = 1250` kayıt içerir. Önceden hesaplanan 1000 kaydın uygun global cache'leri yeniden kullanılır; 50 reserve kayıt çağrılmaz. Her model için yeni inference beklenen sayı 250'dir; model/prompt/cache konfigürasyonu değişmişse cache reuse daha düşük olabilir.

**Önemli:** Stage 04–06 export'larında 1250 soru/model bulunacak; bunların **yalnız yeni seçilmiş 250 tanesi** final testtir. Geliştirme için `validated_v007`/Stage 10–11 dondurulmuştur; bu yeni 250 kayıt calibrator eğitimine, feature seçimine veya hiperparametre aramasına eklenemez. `validated_v012` değiştirilmez. Bu notebook LLM çağrısı yapmaz, manuel doğruluk etiketi okumaz ve test skorlarını hesaplamaz.

İki aşamalıdır: önce `PREPARE_REVIEW` ile kaynak/zorluk formunu oluştur; mevcut `validated_v012` kayıtlarında kaynaklar ve zorluk önceden doldurulmuşsa form otomatik devralınır. Eksikleri insan değerlendirmesiyle doldur. Sonra `RUN_FREEZE=True` ve manuel onayla final dosyalarını **bir kere** oluştur.

## 1. Drive / imports

In [1]:
from google.colab import drive
drive.mount('/content/drive')

import csv, hashlib, json, re, unicodedata, math
from collections import Counter
from datetime import datetime, timezone
from pathlib import Path
from urllib.parse import urlparse

import numpy as np
import pandas as pd
from scipy.optimize import milp, Bounds, LinearConstraint

print('SciPy MILP: AVAILABLE')

Mounted at /content/drive
SciPy MILP: AVAILABLE


## 2. Yollar ve çalışma modu

İlk çalıştırmada iki onay bayrağını **False** bırak. Kaynak ve zorluk formunu kaydettikten ve doğruladıktan sonra, yalnız final test inference başlamamışsa `RUN_FREEZE=True` ve `MANUAL_REVIEW_CONFIRMED=True` yap.

In [11]:
PROJECT_ROOT = Path('/content/drive/MyDrive/LLM_Confidence_Thesis')
DEV_VERSION = 'validated_v007'
PARENT_VERSION = 'validated_v012'
INFERENCE_DATASET_VERSION = 'validated_v012_final250_v1'

DEV_PATH = PROJECT_ROOT / 'data' / 'validated' / f'{DEV_VERSION}.json'
PARENT_PATH = PROJECT_ROOT / 'data' / 'validated' / f'{PARENT_VERSION}.json'
PARENT_MANIFEST_PATH = PROJECT_ROOT / 'data' / 'validated' / f'{PARENT_VERSION}_manifest.json'
PROTO_DIR = PROJECT_ROOT / 'results' / 'stage12' / DEV_VERSION
PROTOCOL_PATH = PROTO_DIR / 'final_test_protocol_v2.json'
STAGE12_MANIFEST_PATH = PROTO_DIR / 'stage12_protocol_manifest_v2.json'

OUTPUT_DIR = PROJECT_ROOT / 'results' / 'stage13' / INFERENCE_DATASET_VERSION
REVIEW_CSV = OUTPUT_DIR / 'candidate_source_difficulty_review_v1.csv'
QUESTIONS_PATH = OUTPUT_DIR / 'final_test_questions_v1.json'
GOLD_PATH = OUTPUT_DIR / 'final_test_gold_v1.json'
SPLIT_PATH = OUTPUT_DIR / 'candidate_split_registry_v1.json'
STAGE13_MANIFEST_PATH = OUTPUT_DIR / 'stage13_final_test_manifest_v1.json'
DERIVED_DATA_PATH = PROJECT_ROOT / 'data' / 'validated' / f'{INFERENCE_DATASET_VERSION}.json'
DERIVED_MANIFEST_PATH = PROJECT_ROOT / 'data' / 'validated' / f'{INFERENCE_DATASET_VERSION}_manifest.json'

RUN_FREEZE = True
MANUAL_REVIEW_CONFIRMED = True  # Doğruluk, kaynaklar, aliases ve semantic dedup incelendiğinde True.

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
print('MODE:', 'FREEZE' if RUN_FREEZE else 'PREPARE_REVIEW')
print('Stage13:', OUTPUT_DIR)

MODE: FREEZE
Stage13: /content/drive/MyDrive/LLM_Confidence_Thesis/results/stage13/validated_v012_final250_v1


## 3. Hash yardımcıları / immutable artifact kuralı

In [3]:
def sha256_file(path):
    h = hashlib.sha256()
    with open(path, 'rb') as f:
        for chunk in iter(lambda: f.read(1024 * 1024), b''):
            h.update(chunk)
    return h.hexdigest()

def read_json(path):
    if not path.is_file():
        raise FileNotFoundError(path)
    with open(path, 'r', encoding='utf-8') as f:
        return json.load(f)

def write_json_immutable(path, data):
    raw = (json.dumps(data, ensure_ascii=False, indent=2, allow_nan=False) + '\n').encode('utf-8')
    path.parent.mkdir(parents=True, exist_ok=True)
    if path.exists():
        if path.read_bytes() != raw:
            raise FileExistsError(f'IMMUTABLE CONFLICT: {path} — var olanı silmeyin/üzerine yazmayın.')
        print('Immutable artifact already identical:', path)
    else:
        with open(path, 'xb') as f:
            f.write(raw)
        print('Created immutable artifact:', path)
    return sha256_file(path)

def assert_unique_ids(rows, name):
    ids = [r['question_id'] for r in rows]
    if len(ids) != len(set(ids)):
        raise RuntimeError(f'{name}: duplicate question_id')

def normalize_question(text):
    value = unicodedata.normalize('NFKC', str(text)).casefold()
    value = ''.join(ch if ch.isalnum() else ' ' for ch in value)
    return ' '.join(value.split())

def require(condition, message):
    if not condition:
        raise RuntimeError(message)

## 4. Stage 12 sözleşmesini ve `validated_v012` bütünlüğünü doğrula

In [4]:
protocol = read_json(PROTOCOL_PATH)
stage12_manifest = read_json(STAGE12_MANIFEST_PATH)
dev = read_json(DEV_PATH)
parent = read_json(PARENT_PATH)
parent_manifest = read_json(PARENT_MANIFEST_PATH)

require(stage12_manifest['protocol_sha256'] == sha256_file(PROTOCOL_PATH), 'Stage12 protocol SHA mismatch')
require(protocol['status'] == 'LOCKED_PROTOCOL_NO_FINAL_TEST_EVALUATION', 'Stage12 kilit durumu uyumsuz')
require(protocol['protocol_version'] == 'final_test_protocol_v2', 'Stage12 protocol sürümü uyumsuz')
require(protocol['validated_dataset_sha256'] == sha256_file(DEV_PATH), 'Dondurulmuş validated_v007 SHA mismatch')
require(parent_manifest['validated_file_sha256'] == sha256_file(PARENT_PATH), 'validated_v012 SHA mismatch')
require(parent_manifest['n_validated_questions'] == 1300, 'Parent manifest n != 1300')
require(parent_manifest['source_raw_version'] == 'raw_v012', 'Parent source raw mismatch')
require(len(dev) == 1000 and len(parent) == 1300, 'Beklenen 1000/1300 veri boyutları uyuşmuyor')
assert_unique_ids(dev, 'DEV')
assert_unique_ids(parent, 'PARENT')

old_by_id = {r['question_id']:r for r in dev}
parent_by_id = {r['question_id']:r for r in parent}
old_ids = set(old_by_id)
new_ids = set(parent_by_id) - old_ids
require(old_ids.issubset(parent_by_id), 'validated_v007 içinden silinmiş ID var')
require(len(new_ids) == 300, f'Yeni aday sayısı 300 değil: {len(new_ids)}')
IMMUTABLE_FIELDS = ('question','reference_answer','accepted_aliases','answer_type','category')
for qid, old in old_by_id.items():
    for field in IMMUTABLE_FIELDS:
        require(old.get(field) == parent_by_id[qid].get(field), f'Eski soru içeriği değişmiş: {qid}/{field}')
for r in parent:
    require(r['verification_status'] == 'verified', f'Unverified kayıt: {r["question_id"]}')

candidate_rows = sorted((parent_by_id[q] for q in new_ids), key=lambda r:r['question_id'])
old_texts = {normalize_question(r['question']) for r in dev}
new_texts = [normalize_question(r['question']) for r in candidate_rows]
require(len(set(new_texts)) == len(new_texts), 'Yeni adaylar arasında exact-normalized metin tekrarları var')
require(not old_texts.intersection(new_texts), 'Eski ve yeni arasında exact-normalized metin tekrarı var')

registry = protocol['final_test_registry']
require(registry['n_independent_questions'] == 250, 'Stage12 final N !=250')
require(registry['n_candidate_questions_initial_target'] == 300, 'Stage12 candidate N !=300')
category_quota = registry['category_final_quotas']
type_quota = registry['answer_type_final_quotas']
difficulty_quota = registry['difficulty_preinference_target_counts']
require(sum(category_quota.values()) == sum(type_quota.values()) == sum(difficulty_quota.values()) == 250, 'Stage12 kotaları toplamı !=250')
require(set(r['category'] for r in candidate_rows).issubset(category_quota), 'Beklenmeyen category')
require(set(r['answer_type'] for r in candidate_rows).issubset(type_quota), 'Beklenmeyen answer_type')

print('STAGE 12 LOCK + V007/V012 SHA + 1000 IMMUTABLE + 300 NEW IDS: PASS')
print('Old records:', len(dev), '| New candidates:', len(candidate_rows))
print('Category quotas:', category_quota)
print('Answer-type quotas:', type_quota)
print('Difficulty quotas:', difficulty_quota)

STAGE 12 LOCK + V007/V012 SHA + 1000 IMMUTABLE + 300 NEW IDS: PASS
Old records: 1000 | New candidates: 300
Category quotas: {'Bilim': 20, 'Coğrafya': 19, 'Dil': 20, 'Edebiyat': 19, 'Ekonomi': 19, 'Felsefe': 19, 'Genel Kültür': 19, 'Matematik': 19, 'Sanat': 20, 'Spor': 19, 'Tarih': 19, 'Teknoloji': 19, 'Uzay': 19}
Answer-type quotas: {'BINARY': 15, 'DATE': 33, 'ENTITY_LOCATION': 42, 'ENTITY_ORGANIZATION': 25, 'ENTITY_OTHER': 25, 'ENTITY_PERSON': 43, 'NUMBER': 25, 'NUMBER_WITH_UNIT': 17, 'SHORT_PHRASE': 25}
Difficulty quotas: {'easy': 75, 'hard': 50, 'medium': 125}


## 5. Ön-inference kapısı: yeni adaylar için Stage 04 cache / export var mı?

Yeni 300 soru hakkında önceden üretilmiş hedef-model çıktısı varsa, seçim bu çıktılardan etkilenmiş olabileceğinden dondurma işlemini otomatik durdurur. Henüz freeze yapmıyorsan tarama bilgi amaçlıdır.

In [5]:
unexpected = []
stage04_cache_root = PROJECT_ROOT / 'runs' / 'main_answers' / 'cache'
if stage04_cache_root.exists():
    for cache_path in sorted(stage04_cache_root.glob('*/*.jsonl')):
        with cache_path.open('r', encoding='utf-8') as f:
            for line_no, line in enumerate(f, 1):
                if not line.strip():
                    continue
                row = json.loads(line)
                if row.get('question_id') in new_ids:
                    unexpected.append((str(cache_path), line_no, row['question_id']))
                    if len(unexpected) >= 10:
                        break
        if len(unexpected) >= 10:
            break

if unexpected:
    print('WARNING: New candidate IDs already exist in Stage04 cache:', unexpected[:10])
else:
    print('PRE-INFERENCE STAGE04 CACHE CHECK: PASS (new candidate ID not found)')

require(not (RUN_FREEZE and unexpected), 'Stage04 cache contains target outputs for new candidates; do not freeze automatically')
require(not (RUN_FREEZE and (PROJECT_ROOT / 'results' / 'stage04' / 'validated_v012_stage04_complete.json').exists()),
        'Full v012 Stage04 run exists; review the pre-inference protocol before freezing.')

PRE-INFERENCE STAGE04 CACHE CHECK: PASS (new candidate ID not found)


## 6. Kaynak + difficulty inceleme formunu hazırla

Stage 12 v2, final teste alınacak sorularda **izlenebilir kaynak adı ve referansı** ile **model çıktısı görülmeden atanmış easy/medium/hard zorluk** hedefleri istiyor. Eski kayıtların `verification_source='MANUEL'` değeri tek başına kaynak künyesi değildir.

İlk çalıştırmada CSV oluşturulur; mevcutsa **asla üzerine yazılmaz**. Google Sheets/Excel'de şu alanları tamamla: `source_name`, `source_reference`, `difficulty`. `eligible=0` yaparak geçersiz adayı çıkarabilirsin; `rejection_reason` doldur. Bilgi veya cevap hatalıysa raw JSON'u sessizce değiştirme; önceden kayıtlı doğrulama sürüm sürecini uygula. Kaynaklar olgusal olarak manuel kontrol edilmelidir; notebook yalnız biçimi kontrol eder.

In [6]:
REVIEW_COLUMNS = [
    'question_id','question','category','answer_type','reference_answer',
    'source_name','source_reference','difficulty','eligible','rejection_reason',
]
if not REVIEW_CSV.exists():
    rows = []
    for r in candidate_rows:
        validation = r.get('validation', {})
        source_name = str(r.get('source_name') or '').strip()
        source_ref = str(r.get('source_reference') or '').strip()
        # Yalnız gerçekten izlenebilir bir referanssa validator field'ından devral.
        review_ref = str(validation.get('verification_source') or '').strip()
        if not source_ref and review_ref and review_ref.upper() not in {'MANUEL','MANUAL','VERIFIED','NONE','N/A','NA','-'}:
            source_ref = review_ref
        if not source_name and source_ref.startswith(('http://','https://')):
            source_name = urlparse(source_ref).netloc
        rows.append({
            'question_id':r['question_id'], 'question':r['question'],
            'category':r['category'], 'answer_type':r['answer_type'],
            'reference_answer':r['reference_answer'],
            'source_name':source_name, 'source_reference':source_ref,
            'difficulty':str(r.get('difficulty') or '').strip().lower(),
            'eligible':'1', 'rejection_reason':'',
        })
    with REVIEW_CSV.open('x',encoding='utf-8-sig',newline='') as f:
        writer=csv.DictWriter(f,fieldnames=REVIEW_COLUMNS)
        writer.writeheader()
        writer.writerows(rows)
    print('Created review form:', REVIEW_CSV)
else:
    print('Existing review form preserved:', REVIEW_CSV)
print('Do NOT inspect target-model answers before freeze.')

Created review form: /content/drive/MyDrive/LLM_Confidence_Thesis/results/stage13/validated_v012_final250_v1/candidate_source_difficulty_review_v1.csv
Do NOT inspect target-model answers before freeze.


## 7. Review formunu denetle ve kotaların seçilebilirliğini raporla

In [9]:
review_df = pd.read_csv(REVIEW_CSV,dtype=str,keep_default_na=False,encoding='utf-8-sig')
require(list(review_df.columns) == REVIEW_COLUMNS, 'Review CSV kolonları beklenenden farklı')
require(len(review_df) == 300, 'Review CSV tam 300 satır içermeli')
require(review_df['question_id'].is_unique, 'Review CSV duplicate ID')
require(set(review_df['question_id']) == new_ids, 'Review CSV candidate ID set mismatch')

candidate_lookup = {r['question_id']:r for r in candidate_rows}
for row in review_df.to_dict('records'):
    orig=candidate_lookup[row['question_id']]
    for col in ['question','category','answer_type','reference_answer']:
        require(row[col] == str(orig[col]), f'Review formunda protected field değişmiş: {row["question_id"]}/{col}')

INVALID_SOURCES = {'','manuel','manual','verified','none','n/a','na','-','unknown'}
def source_valid(name, ref):
    return bool(str(name).strip()) and str(ref).strip().casefold() not in INVALID_SOURCES and len(str(ref).strip()) >= 8

review_df['eligible_norm'] = review_df['eligible'].str.strip()
require(review_df['eligible_norm'].isin(['0','1']).all(), 'eligible yalnız 0/1 olabilir')
review_df['difficulty_norm'] = review_df['difficulty'].str.strip().str.lower()
review_df['source_ready'] = [source_valid(n,r) for n,r in zip(review_df['source_name'],review_df['source_reference'])]
review_df['difficulty_ready'] = review_df['difficulty_norm'].isin(['easy','medium','hard'])
review_df['selection_eligible'] = (review_df['eligible_norm']=='1') & review_df['source_ready'] & review_df['difficulty_ready']
require(not ((review_df['eligible_norm']=='0') & (review_df['rejection_reason'].str.strip()=='')).any(),
        'eligible=0 satırlarda rejection_reason doldurun')

eligible_df = review_df.loc[review_df['selection_eligible']].copy().sort_values('question_id')
print('Candidate review report:')
print('   300 candidates:',len(review_df))
print('   source ready:',int(review_df['source_ready'].sum()))
print('   difficulty ready:',int(review_df['difficulty_ready'].sum()))
print('   eligible with BOTH:',len(eligible_df))
print('   required final N:',registry['n_independent_questions'])
for label, col, quota in [('Category','category',category_quota),('Answer type','answer_type',type_quota),('Difficulty','difficulty_norm',difficulty_quota)]:
    counts=eligible_df[col].value_counts().to_dict()
    shortages={k:(int(v),int(counts.get(k,0))) for k,v in quota.items() if counts.get(k,0)<v}
    print(label,'individual-quota shortages:',shortages or 'none')
print('Review CSV:', REVIEW_CSV)

Candidate review report:
   300 candidates: 300
   source ready: 300
   difficulty ready: 300
   eligible with BOTH: 300
   required final N: 250
Category individual-quota shortages: none
Answer type individual-quota shortages: none
Difficulty individual-quota shortages: {'easy': (75, 14), 'hard': (50, 41)}
Review CSV: /content/drive/MyDrive/LLM_Confidence_Thesis/results/stage13/validated_v012_final250_v1/candidate_source_difficulty_review_v1.csv


## 8. Final seçim + SHA freeze (yalnız RUN_FREEZE=True ise)

Tam 250 kayıt seçimi `scipy.optimize.milp` ile kategori, cevap türü ve zorluk kotalarını **eşzamanlı** karşılar; eşitlikler değiştirilemez. Çoklu uygun çözüm varsa Stage 12'deki sabit seed, önceden belirlenmiş rastgele amaç katsayılarını üretir. Final soruları **accuracy/confidence/LLM çıktılarına göre seçilmez**.

`MANUAL_REVIEW_CONFIRMED=True`, 300 adayın kaynak/alias/doğruluk ve anlamsal tekrar incelemesini hedef LLM çıktılarını görmeden tamamladığını ifade eder. Etiketleri ve test tahminlerini açmayın.

In [12]:
if not RUN_FREEZE:
    print('PREPARE_REVIEW COMPLETE — freeze yapılmadı. CSV kaynak/zorluk alanlarını doldurun; sonrasında RUN_FREEZE=True.')
else:
    require(MANUAL_REVIEW_CONFIRMED, 'Freeze öncesi insan tarafından kalite + semantic dedup teyidi gerekli')
    require(len(eligible_df) >= 250, f'250 uygun (source+difficulty) aday yok; mevcut={len(eligible_df)}')
    # Guard against pre-inference output after a previous local notebook run.
    require(not unexpected, 'Yeni adaylar için Stage04 cache saptandı')
    n=len(eligible_df)
    rng = np.random.default_rng(int(registry['final_candidate_selection_seed']))
    objective = rng.uniform(0.0, 1.0, n)
    constraints=[]
    for col, quota in [('category', category_quota), ('answer_type', type_quota),]:
        for value, required_n in sorted(quota.items()):
            row = (eligible_df[col].to_numpy()==value).astype(float)
            constraints.append((row, int(required_n)))
    A=np.vstack([row for row,_ in constraints])
    b=np.array([count for _,count in constraints],dtype=float)
    opt=milp(c=objective, integrality=np.ones(n), bounds=Bounds(np.zeros(n),np.ones(n)),
             constraints=LinearConstraint(A,b,b), options={'time_limit':120,'mip_rel_gap':0.0})
    require(opt.status == 0 and opt.x is not None,
            'Joint 3-way quotas infeasible/solver incomplete. Kaynak ve zorluk dağılımını kontrol edin; protokolü sessizce gevşetmeyin. Solver: '+str(opt.message))
    chosen=eligible_df.loc[opt.x>0.5].copy().sort_values('question_id')
    require(len(chosen)==250, 'MILP selected N !=250')
    for col, quota in [('category', category_quota), ('answer_type', type_quota),]:
        require(chosen[col].value_counts().to_dict() == dict(quota), f'Joint quota mismatch: {col}')
    selected_ids=set(chosen['question_id'])
    require(len(selected_ids)==250 and not selected_ids.intersection(old_ids),'Test/dev ID overlap')
    candidate_difficulty_distribution = {
        level: int((eligible_df['difficulty_norm'] == level).sum())
        for level in ('easy', 'medium', 'hard')
    }

    selected_difficulty_distribution = {
        level: int((chosen['difficulty_norm'] == level).sum())
        for level in ('easy', 'medium', 'hard')
    }

    require(
        sum(selected_difficulty_distribution.values()) == 250,
        'Selected difficulty distribution N != 250'
    )

    print('Candidate difficulty:', candidate_difficulty_distribution)
    print('Selected difficulty:', selected_difficulty_distribution)
    # Preserve all 1000 old (including 30 Stage03 sanity IDs): Stage04's existing audit requires them.
    sanity=set(read_json(PROJECT_ROOT / 'results' / 'model_selection' / 'selected_models_v1.json')['sanity_question_ids'])
    require(len(sanity)==30 and sanity.issubset(old_ids),'Stage03 30 immutable sanity questions absent from original 1000')
    review_lookup = review_df.set_index('question_id').to_dict('index')
    dataset_out = []
    for source_row in parent:
        qid = source_row['question_id']
        if qid not in old_ids and qid not in selected_ids:
            continue
        # Eski 1000 kayıt alanlarına dokunma; seçili test sorularına yalnızca
        # ayrıca doğrulanmış kaynak meta-verisini ekle (soru/gold değiştirme).
        item = dict(source_row)
        if qid in selected_ids:
            item['source_name'] = review_lookup[qid]['source_name'].strip()
            item['source_reference'] = review_lookup[qid]['source_reference'].strip()
        dataset_out.append(item)
    require(len(dataset_out)==1250 and len({r['question_id'] for r in dataset_out})==1250,'Inference dataset N/uniqueness')
    require([r['question_id'] for r in dataset_out if r['question_id'] in old_ids]==[r['question_id'] for r in dev],
            'Original 1000 row ordering changed')
    questions_out=[{
        'question_id':qid, 'question':candidate_lookup[qid]['question'],
        'category':candidate_lookup[qid]['category'], 'answer_type':candidate_lookup[qid]['answer_type']
    } for qid in sorted(selected_ids)]
    gold_out=[{
        'question_id':qid, 'reference_answer':candidate_lookup[qid]['reference_answer'],
        'accepted_aliases':candidate_lookup[qid]['accepted_aliases'],
        'source_name':review_lookup[qid]['source_name'].strip(),
        'source_reference':review_lookup[qid]['source_reference'].strip(),
        'difficulty':review_lookup[qid]['difficulty_norm'],
    } for qid in sorted(selected_ids)]
    split_out=[{
        'question_id':r['question_id'],
        'dataset_split':'FINAL_TEST' if r['question_id'] in selected_ids else 'RESERVE',
        'category':r['category'],'answer_type':r['answer_type'],
        'difficulty':review_lookup[r['question_id']]['difficulty_norm'] or None,
        'source_review_complete':bool(review_lookup[r['question_id']]['source_ready']),
        'rejection_reason':review_lookup[r['question_id']]['rejection_reason'].strip() or None,
    } for r in candidate_rows]
    require(sum(r['dataset_split']=='FINAL_TEST' for r in split_out)==250,'Final split N mismatch')

    # Pre-inference protocol amendment:
    # Stage12 v2 dosyası değiştirilmez; zorluk kotasından sapma ayrıca kaydedilir.

    AMENDMENT_PATH = (
        STAGE13_MANIFEST_PATH.parent
        / 'stage12_v2_difficulty_amendment_v1.json'
    )

    difficulty_amendment = {
        'amendment_version': 'stage12_v2_difficulty_amendment_v1',
        'original_stage12_protocol_path': str(PROTOCOL_PATH),
        'original_stage12_protocol_sha256': sha256_file(PROTOCOL_PATH),

        'reason': (
            'Validated 300-candidate pool cannot satisfy the originally '
            'planned difficulty quotas before model inference: '
            'easy requires 75 but has 14; hard requires 50 but has 41.'
        ),

        'original_difficulty_quotas': dict(difficulty_quota),
        'candidate_difficulty_distribution': candidate_difficulty_distribution,

        'effective_selection_constraints': [
            'category',
            'answer_type',
        ],
        'difficulty_selection_rule': 'descriptive_only',
        'selected_difficulty_distribution': selected_difficulty_distribution,

        'final_test_size_unchanged': 250,
        'category_quotas_unchanged': dict(category_quota),
        'answer_type_quotas_unchanged': dict(type_quota),

        'amendment_made_before_target_model_inference': True,
        'target_model_test_outputs_inspected_for_selection': False,
    }

    amendment_sha = write_json_immutable(
        AMENDMENT_PATH,
        difficulty_amendment
    )

    # Final TEST register+gold files remain physically distinct from the model-inference-only registry.
    qsha=write_json_immutable(QUESTIONS_PATH,questions_out)
    gsha=write_json_immutable(GOLD_PATH,gold_out)
    ssha=write_json_immutable(SPLIT_PATH,split_out)
    dsha=write_json_immutable(DERIVED_DATA_PATH,dataset_out)
    derived_manifest={
        'validated_dataset_version':INFERENCE_DATASET_VERSION,
        'validated_dataset_file':DERIVED_DATA_PATH.name,
        'source_raw_version':parent_manifest['source_raw_version'],
        'source_validated_version':PARENT_VERSION,
        'source_validated_sha256':sha256_file(PARENT_PATH),
        'source_raw_file_sha256':parent_manifest['source_raw_file_sha256'],
        'subset_rule':'All 1000 immutable validated_v007 + 250 selected final_test, exclude 50 reserve; only selected 250 are final test.',
        'n_raw_questions':parent_manifest['n_raw_questions'],
        'n_validated_questions':len(dataset_out),
        'n_excluded_questions':50, # Bunlar gecersiz degil, alt-kumeye alinmayan RESERVE kayitlaridir.
        'n_reserved_not_in_inference_dataset':50,
        'validated_file_sha256':dsha,
        'category_distribution':dict(sorted(Counter(r['category'] for r in dataset_out).items())),
        'answer_type_distribution':dict(sorted(Counter(r['answer_type'] for r in dataset_out).items())),
        'stage12_protocol_sha256':sha256_file(PROTOCOL_PATH),
        'stage13_final_questions_sha256':qsha,
    }
    dmsha=write_json_immutable(DERIVED_MANIFEST_PATH,derived_manifest)
    stage13_manifest={
        'stage':'13_final_test_registry', 'version':'stage13_final_test_v1',
        'stage12_protocol_path':str(PROTOCOL_PATH),
        'stage12_protocol_sha256':sha256_file(PROTOCOL_PATH),
        'parent_validated_dataset':str(PARENT_PATH),
        'parent_validated_sha256':sha256_file(PARENT_PATH),
        'old_dev_validated_dataset':str(DEV_PATH),
        'old_dev_validated_sha256':sha256_file(DEV_PATH),
        'candidate_review_file':str(REVIEW_CSV), 'candidate_review_sha256':sha256_file(REVIEW_CSV),
        'candidate_count':300,'n_final_questions':250,'n_reserve_questions':50,
        'seed':int(registry['final_candidate_selection_seed']),
        'category_quotas':dict(category_quota),'answer_type_quotas':dict(type_quota),
        'difficulty_quotas_original_not_enforced': dict(difficulty_quota),
        'difficulty_quotas_enforced': False,
        'difficulty_selection_rule': 'descriptive_only',
        'candidate_difficulty_distribution': candidate_difficulty_distribution,
        'selected_difficulty_distribution': selected_difficulty_distribution,
        'selection_protocol_amendment_path': str(AMENDMENT_PATH),
        'selection_protocol_amendment_sha256': amendment_sha,
        'final_question_ids':sorted(selected_ids),
        'final_test_questions_path':str(QUESTIONS_PATH),'final_test_questions_sha256':qsha,
        'final_test_gold_path':str(GOLD_PATH),'final_test_gold_sha256':gsha,
        'split_registry_path':str(SPLIT_PATH),'split_registry_sha256':ssha,
        'stage04_06_compatible_inference_dataset':str(DERIVED_DATA_PATH),
        'inference_dataset_sha256':dsha,
        'inference_dataset_manifest':str(DERIVED_MANIFEST_PATH),
        'inference_dataset_manifest_sha256':dmsha,
        'stage03_sanity_questions_retained':30,
        'manual_quality_and_semantic_dedup_confirmed_before_target_inference':True,
        'target_model_test_outputs_inspected_for_selection':False,
        'test_manual_correctness_labels_inspected':False,
        'status':'FROZEN_PREINFERENCE',
    }
    msha=write_json_immutable(STAGE13_MANIFEST_PATH,stage13_manifest)
    print('STAGE 13 FINAL TEST: FROZEN PRE-INFERENCE')
    print('Selected test:',len(selected_ids),'| reserve:',len(new_ids-selected_ids))
    print('Inference dataset:',DERIVED_DATA_PATH,'| n=',len(dataset_out))
    print('Selected question/gold SHA:',qsha,gsha)
    print('Stage13 manifest SHA:',msha)
    print('STAGE04/05/06 VERSION:',INFERENCE_DATASET_VERSION)
    print('Do not rerun selection with another seed or modify frozen files.')

Candidate difficulty: {'easy': 14, 'medium': 245, 'hard': 41}
Selected difficulty: {'easy': 8, 'medium': 209, 'hard': 33}
Created immutable artifact: /content/drive/MyDrive/LLM_Confidence_Thesis/results/stage13/validated_v012_final250_v1/stage12_v2_difficulty_amendment_v1.json
Created immutable artifact: /content/drive/MyDrive/LLM_Confidence_Thesis/results/stage13/validated_v012_final250_v1/final_test_questions_v1.json
Created immutable artifact: /content/drive/MyDrive/LLM_Confidence_Thesis/results/stage13/validated_v012_final250_v1/final_test_gold_v1.json
Created immutable artifact: /content/drive/MyDrive/LLM_Confidence_Thesis/results/stage13/validated_v012_final250_v1/candidate_split_registry_v1.json
Created immutable artifact: /content/drive/MyDrive/LLM_Confidence_Thesis/data/validated/validated_v012_final250_v1.json
Created immutable artifact: /content/drive/MyDrive/LLM_Confidence_Thesis/data/validated/validated_v012_final250_v1_manifest.json
Created immutable artifact: /content/dr

## 9. Bundan sonraki Stage 04–07 akışı ve test izolasyonu

**Stage 04–06:** Mevcut kodlarını değiştirme; yalnız her birinin mevcut `VALIDATED_DATASET_VERSION` ayarını `validated_v012_final250_v1` yap. Stage 04'te `Dataset N: 1250`, uygun aynı global cache kullanılmışsa `Cache reuse: 1000 / New inference: 250` görmelisin; model, prompt veya config hash farklıysa reuse garanti değildir. 50 RESERVE için yeni inference yapılmaz. Stage 04'ün 30 Stage-03 sanity sorusu audit'i **geçer**, çünkü eski 1000'in tamamı aynen tutuldu.

**Stage 07 uyarısı:** Mevcut Stage 07, bu dataset'in tamamına (1000 eski + 250 yeni) karşı manuel annotation şablonu üretir; eski 3000 model cevabını sıfırdan yeniden etiketlemek zorunda kalmaman için Stage 07'yi başlatmadan önce önceki etiketleri **aynı question_id, model_id ve frozen answer hash** ile güvenli devralma yolunu kurmamız gerekir. Alternatif olarak yalnız yeni 750 final-test model cevabını etiketleyen izole bir aşama oluştururuz. Hiçbir final test etiketi calibrator eğitimine dahil edilmez.

**Final-stage feature assembly:** Stage 08'in eski tüm-dataset pipeline'ını direkt olarak yeni eğitim/veri seçimi amacıyla çalıştırmayın. `validated_v007` CLEAN_DEV (970/model) calibrator eğitiminin tek kaynağıdır. Final-test prediction dosyalarının SHA'ları test etiketlerini açmadan önce dondurulacaktır.

**Olgusal kaynak notu:** Stage 02 `ACCEPT` kararı, tek başına dış kaynak künyesi sağlamaz. Stage 12 v2'nin gerektirdiği kaynakları `candidate_source_difficulty_review_v1.csv` içinde açıklanabilir şekilde doldur; dosyanın Stage 13 manifestine hash'i bağlanır. Yeni adaylarla eski 1000 arasındaki normalize metin karşılaştırması yalnız exact duplicate denetimidir; semantic duplicate denetimi manuel beyanla kayıt altına alınır.